# LLMOps Assignment 2 — API Cost, Latency & Prompt Version Management
### Course: MLOps | EduAssist AI Academic Q&A Assistant
---
| Item | Value |
|---|---|
| **Provider** | Groq (OpenAI-compatible API) |
| **Model** | `openai/gpt-oss-20b` |
| **MLflow Experiment** | `eduassist_llmops_assignment` |
| **Pricing Date** | 2026-11-03 (USD) |
| **Input price** | $0.075 / 1M tokens |
| **Output price** | $0.30 / 1M tokens |

> Free-tier path: create a key at https://console.groq.com (no OpenAI billing required).
> Costs below use Groq's **published** on-demand rates for assignment economics.

**Tasks**
- **Task 1 (2 marks):** Call the LLM API and measure latency for 5 academic questions
- **Task 2 (3 marks):** Extract token usage and calculate per-request and monthly costs
- **Task 3 (5 marks):** Prompt versioning, MLflow evaluation, comparison, and production promotion


In [18]:
# ── Install required packages ─────────────────────────────────
import subprocess, sys

PACKAGES = [
    "openai>=1.0.0",   # Groq exposes an OpenAI-compatible endpoint
    "mlflow>=2.14.0",
    "pandas",
    "numpy",
    "python-dotenv",
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✓ All packages installed successfully")


✓ All packages installed successfully


## Setup — Imports, Configuration, and API Client

In [1]:
import os, time, json, statistics, warnings
import pandas as pd
import numpy as np
from datetime import datetime
from openai import OpenAI
import mlflow
from mlflow import MlflowClient

warnings.filterwarnings("ignore")

# Load .env if present (never hard-code secrets)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Configuration ────────────────────────────────────────────────
# Groq free-tier key from https://console.groq.com/keys
# Uses OpenAI-compatible Chat Completions API.
PROVIDER                 = "groq"
MODEL_NAME               = "openai/gpt-oss-20b"
GROQ_BASE_URL            = "https://api.groq.com/openai/v1"
# Published Groq on-demand pricing (USD / 1M tokens). Free-tier keys
# may not be billed the same way; we still use published rates for the
# assignment's cost formulas. Source: https://groq.com/pricing (2026-11-03)
INPUT_PRICE_PER_MILLION  = 0.075
OUTPUT_PRICE_PER_MILLION = 0.30
PRICING_DATE             = "2026-11-03"
PRICING_CURRENCY         = "USD"
MONTHLY_REQUESTS         = 10_000
MLFLOW_TRACKING_URI      = "http://127.0.0.1:5000"
EXPERIMENT_NAME          = "eduassist_llmops_assignment"
PROMPT_REGISTRY_NAME     = "eduassist_prompt"

# ── Groq client (key loaded from GROQ_API_KEY env var) ───────────
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY is not set.\n"
        "1. Create a free key at https://console.groq.com/keys\n"
        "2. Export it with:  export GROQ_API_KEY='gsk_...'\n"
        "Or add it to a .env file in this directory."
    )

client = OpenAI(api_key=api_key, base_url=GROQ_BASE_URL)

print(f"✓ Provider        : {PROVIDER}")
print(f"✓ Model           : {MODEL_NAME}")
print(f"✓ MLflow version  : {mlflow.__version__}")
print(f"✓ API key loaded  : yes (GROQ_API_KEY)")
print(f"✓ Input price     : ${INPUT_PRICE_PER_MILLION}/1M tokens  ({PRICING_DATE}, {PRICING_CURRENCY})")
print(f"✓ Output price    : ${OUTPUT_PRICE_PER_MILLION}/1M tokens ({PRICING_DATE}, {PRICING_CURRENCY})")


✓ Provider        : groq
✓ Model           : openai/gpt-oss-20b
✓ MLflow version  : 3.17.2
✓ API key loaded  : yes (GROQ_API_KEY)
✓ Input price     : $0.075/1M tokens  (2026-11-03, USD)
✓ Output price    : $0.3/1M tokens (2026-11-03, USD)


In [2]:
# ── Five academic questions ──────────────────────────────────────
QUESTIONS = [
    {"id": "Q1", "question": "What is photosynthesis and why is it important for life on Earth?"},
    {"id": "Q2", "question": "Explain the Pythagorean theorem and provide a real-world example."},
    {"id": "Q3", "question": "What is Newton's second law of motion and how is it applied in everyday life?"},
    {"id": "Q4", "question": "What is the difference between mitosis and meiosis in cell division?"},
    {"id": "Q5", "question": "Explain the water cycle and describe its main stages."},
]

print(f"{'ID':<5}  Question")
print("-" * 75)
for q in QUESTIONS:
    print(f"{q['id']:<5}  {q['question']}")


ID     Question
---------------------------------------------------------------------------
Q1     What is photosynthesis and why is it important for life on Earth?
Q2     Explain the Pythagorean theorem and provide a real-world example.
Q3     What is Newton's second law of motion and how is it applied in everyday life?
Q4     What is the difference between mitosis and meiosis in cell division?
Q5     Explain the water cycle and describe its main stages.


In [3]:
# ── Helper functions ─────────────────────────────────────────────

def call_llm(prompt_text: str, prompt_version: str = "v1") -> dict:
    """
    Call Groq (OpenAI-compatible) chat completions API and measure wall-clock latency.
    Returns a dict with response, start_time, end_time, latency_seconds, status.
    """
    start_dt  = datetime.now()
    start_ts  = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt_text}],
            temperature=0.7,
        )
        latency = time.perf_counter() - start_ts
        return {
            "response":        response,
            "start_time":      start_dt.isoformat(),
            "end_time":        datetime.now().isoformat(),
            "latency_seconds": round(latency, 4),
            "status":          "success",
        }
    except Exception as exc:
        latency = time.perf_counter() - start_ts
        return {
            "response":        None,
            "start_time":      start_dt.isoformat(),
            "end_time":        datetime.now().isoformat(),
            "latency_seconds": round(latency, 4),
            "status":          f"error: {exc}",
        }


def calculate_cost(input_tokens: int, output_tokens: int) -> tuple:
    """Return (input_cost_usd, output_cost_usd, total_cost_usd)."""
    ic = (input_tokens  / 1_000_000) * INPUT_PRICE_PER_MILLION
    oc = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    return round(ic, 8), round(oc, 8), round(ic + oc, 8)


def validate_json_response(text: str) -> tuple:
    """
    Returns (is_valid: bool, parsed: dict|None, message: str).
    Validates v3 JSON against the required schema:
      {answer: str, confidence: 'high'|'medium'|'low', key_points: [str]}
    """
    try:
        cleaned = text.strip()
        # Strip markdown code fences if present
        if cleaned.startswith("```"):
            lines   = cleaned.split("\n")
            cleaned = "\n".join(
                l for l in lines if not l.startswith("```")
            )
        data = json.loads(cleaned)
        required = {"answer", "confidence", "key_points"}
        missing  = required - data.keys()
        if missing:
            return False, data, f"Missing keys: {missing}"
        if data.get("confidence") not in {"high", "medium", "low"}:
            return False, data, f"Invalid confidence: {data.get('confidence')!r}"
        if not isinstance(data.get("key_points"), list):
            return False, data, "key_points must be a list"
        return True, data, "Valid JSON"
    except json.JSONDecodeError as exc:
        return False, None, f"JSONDecodeError: {exc}"


def render_prompt(template: str, question: str) -> str:
    """Replace {{question}} placeholder with actual question text."""
    return template.replace("{{question}}", question)


print("✓ Helper functions defined: call_llm | calculate_cost | validate_json_response | render_prompt")


✓ Helper functions defined: call_llm | calculate_cost | validate_json_response | render_prompt


---
## Task 1 — Call an LLM and Measure Latency  *(2 marks)*

**Requirements:**
- Load API key from an environment variable (no hard-coding)
- Record: question ID, prompt version, model name, start time, end time, latency, response status
- Calculate mean, median, min, max, standard deviation of latency
- Explain the slowest request


In [4]:
# ── Prompt v1 template (used for Task 1 baseline) ───────────────
PROMPT_V1_TEMPLATE = (
    "Answer the following student question clearly.\n\n"
    "Question: {question}"
)

print("Running Task 1: 5 API calls with Prompt v1")
print("=" * 60)

task1_records = []
for q in QUESTIONS:
    prompt_text = PROMPT_V1_TEMPLATE.format(question=q["question"])
    result      = call_llm(prompt_text, "v1")

    record = {
        "question_id":     q["id"],
        "prompt_version":  "v1",
        "model_name":      MODEL_NAME,
        "start_time":      result["start_time"],
        "end_time":        result["end_time"],
        "latency_seconds": result["latency_seconds"],
        "response_status": result["status"],
    }

    if result["response"]:
        u = result["response"].usage
        record.update({
            "input_tokens":  u.prompt_tokens,
            "output_tokens": u.completion_tokens,
            "total_tokens":  u.total_tokens,
            "response_text": result["response"].choices[0].message.content,
        })
    else:
        record.update({"input_tokens": 0, "output_tokens": 0,
                        "total_tokens": 0, "response_text": ""})

    task1_records.append(record)
    print(f"  {q['id']}: latency={record['latency_seconds']:.3f}s | "
          f"tokens={record['total_tokens']} | status={record['response_status']}")

print(f"\n✓ Task 1 complete — {len(task1_records)} requests recorded")


Running Task 1: 5 API calls with Prompt v1
  Q1: latency=1.418s | tokens=679 | status=success
  Q2: latency=2.144s | tokens=971 | status=success
  Q3: latency=1.281s | tokens=921 | status=success
  Q4: latency=1.216s | tokens=816 | status=success
  Q5: latency=1.383s | tokens=931 | status=success

✓ Task 1 complete — 5 requests recorded


In [5]:
# ── Latency Table and Summary Statistics ─────────────────────────
latencies = [r["latency_seconds"] for r in task1_records
             if r["response_status"] == "success"]

latency_stats = {
    "Mean (s)":    round(statistics.mean(latencies),   4),
    "Median (s)":  round(statistics.median(latencies), 4),
    "Min (s)":     round(min(latencies),               4),
    "Max (s)":     round(max(latencies),               4),
    "Std Dev (s)": round(statistics.stdev(latencies) if len(latencies) > 1 else 0, 4),
}

# Display latency table
df_latency = pd.DataFrame(task1_records)[[
    "question_id", "prompt_version", "model_name",
    "start_time", "end_time", "latency_seconds", "response_status"
]]
print("Latency Table (Task 1):")
print(df_latency.to_string(index=False))

print("\nLatency Summary Statistics:")
for stat, val in latency_stats.items():
    print(f"  {stat:<14}: {val}")

slowest = max(task1_records, key=lambda r: r["latency_seconds"])
fastest = min(task1_records, key=lambda r: r["latency_seconds"])
print(f"\n  Slowest: {slowest['question_id']} — {slowest['latency_seconds']:.3f}s")
print(f"  Fastest: {fastest['question_id']} — {fastest['latency_seconds']:.3f}s")


Latency Table (Task 1):
question_id prompt_version         model_name                 start_time                   end_time  latency_seconds response_status
         Q1             v1 openai/gpt-oss-20b 2026-11-03T20:58:37.708023 2026-11-03T20:58:39.125783           1.4177         success
         Q2             v1 openai/gpt-oss-20b 2026-11-03T20:58:39.125943 2026-11-03T20:58:41.270137           2.1442         success
         Q3             v1 openai/gpt-oss-20b 2026-11-03T20:58:41.270325 2026-11-03T20:58:42.550925           1.2806         success
         Q4             v1 openai/gpt-oss-20b 2026-11-03T20:58:42.551101 2026-11-03T20:58:43.766802           1.2157         success
         Q5             v1 openai/gpt-oss-20b 2026-11-03T20:58:43.766967 2026-11-03T20:58:45.149627           1.3826         success

Latency Summary Statistics:
  Mean (s)      : 1.4882
  Median (s)    : 1.3826
  Min (s)       : 1.2157
  Max (s)       : 2.1442
  Std Dev (s)   : 0.3754

  Slowest: Q2 — 2.144s


In [6]:
# ── Latency Interpretation Paragraph ─────────────────────────────
variability = ("low" if latency_stats["Std Dev (s)"] < 0.5 else "moderate-to-high")
mean_s   = latency_stats['Mean (s)']
median_s = latency_stats['Median (s)']
std_s    = latency_stats['Std Dev (s)']
slow_id  = slowest['question_id']
slow_lat = slowest['latency_seconds']

print(
    f"Latency Interpretation\n"
    f"======================\n"
    f"The five API calls to {MODEL_NAME} recorded a mean latency of\n"
    f"{mean_s:.3f}s (median {median_s:.3f}s) with a standard deviation of\n"
    f"{std_s:.3f}s, indicating {variability} variability across requests.\n\n"
    f"Slowest request: {slow_id} at {slow_lat:.3f}s.\n"
    f"Possible causes:\n"
    f"  1. Longer generated output — questions requiring multi-step explanations\n"
    f"     force the model to decode more tokens, extending wall-clock time\n"
    f"     roughly proportionally to output length.\n"
    f"  2. Server-side load — transient traffic spikes on Groq API can delay\n"
    f"     inference start time independently of prompt complexity.\n"
    f"  3. Network jitter — TCP round-trip variability between the client and\n"
    f"     Groq endpoints adds a small but non-zero overhead per call.\n"
    f"  4. Prompt complexity — a more nuanced question may require more\n"
    f"     attention computation steps before the first token is emitted.\n\n"
    f"The standard deviation of {std_s:.3f}s is acceptable for an academic\n"
    f"Q&A assistant where sub-second precision is not critical."
)


Latency Interpretation
The five API calls to openai/gpt-oss-20b recorded a mean latency of
1.488s (median 1.383s) with a standard deviation of
0.375s, indicating low variability across requests.

Slowest request: Q2 at 2.144s.
Possible causes:
  1. Longer generated output — questions requiring multi-step explanations
     force the model to decode more tokens, extending wall-clock time
     roughly proportionally to output length.
  2. Server-side load — transient traffic spikes on Groq API can delay
     inference start time independently of prompt complexity.
  3. Network jitter — TCP round-trip variability between the client and
     Groq endpoints adds a small but non-zero overhead per call.
  4. Prompt complexity — a more nuanced question may require more
     attention computation steps before the first token is emitted.

The standard deviation of 0.375s is acceptable for an academic
Q&A assistant where sub-second precision is not critical.


---
## Task 2 — Token Counting and Cost Calculation  *(3 marks)*

**Formulae used:**
```
Input Cost  = Input Tokens  / 1,000,000 × $0.075
Output Cost = Output Tokens / 1,000,000 × $0.30
Total Cost  = Input Cost + Output Cost
Monthly Cost (10,000 req) = Average Cost per Request × 10,000
```
Provider: Groq | Model: `openai/gpt-oss-20b` | Pricing date: 2026-11-03 (USD)


In [7]:
# ── Token Usage and Cost per Request ─────────────────────────────
for r in task1_records:
    ic, oc, tc = calculate_cost(r["input_tokens"], r["output_tokens"])
    r["input_cost_usd"]  = ic
    r["output_cost_usd"] = oc
    r["total_cost_usd"]  = tc

df_cost = pd.DataFrame(task1_records)[[
    "question_id", "input_tokens", "output_tokens", "total_tokens",
    "input_cost_usd", "output_cost_usd", "total_cost_usd"
]]

print(f"Model: {MODEL_NAME}")
print(f"Pricing ({PRICING_DATE}, {PRICING_CURRENCY}):")
print(f"  Input tokens : ${INPUT_PRICE_PER_MILLION}/1M")
print(f"  Output tokens: ${OUTPUT_PRICE_PER_MILLION}/1M")
print()
print("Token Usage and Cost per Request:")
print(df_cost.to_string(index=False))


Model: openai/gpt-oss-20b
Pricing (2026-11-03, USD):
  Input tokens : $0.075/1M
  Output tokens: $0.3/1M

Token Usage and Cost per Request:
question_id  input_tokens  output_tokens  total_tokens  input_cost_usd  output_cost_usd  total_cost_usd
         Q1            94            585           679        0.000007         0.000176        0.000183
         Q2            94            877           971        0.000007         0.000263        0.000270
         Q3            97            824           921        0.000007         0.000247        0.000254
         Q4            94            722           816        0.000007         0.000217        0.000224
         Q5            90            841           931        0.000007         0.000252        0.000259


In [8]:
# ── Monthly Cost Projection ──────────────────────────────────────
avg_cost             = df_cost["total_cost_usd"].mean()
monthly_cost         = avg_cost * MONTHLY_REQUESTS
avg_input_tok        = df_cost["input_tokens"].mean()
avg_output_tok       = df_cost["output_tokens"].mean()

print(f"Average cost per request         : ${avg_cost:.8f} {PRICING_CURRENCY}")
print(f"Average input tokens per request : {avg_input_tok:.1f}")
print(f"Average output tokens per request: {avg_output_tok:.1f}")
print()
print(f"Projected Monthly Cost ({MONTHLY_REQUESTS:,} requests):")
print(f"  ${monthly_cost:.4f} {PRICING_CURRENCY}")
print()
print("Cost-Optimization Recommendation:")
print(
    "\n"
    "  1. Cap Output Length  - Adding max_tokens=300 for short factual questions\n"
    "     bounds response length. On Groq, output tokens are priced higher than\n"
    "     input tokens for this model, so shorter answers cut spend directly.\n\n"
    "  2. Shorter System Prompts  - Keep stable role/instruction text minimal.\n"
    "     v3's JSON schema adds input tokens on every call; keep the schema\n"
    "     compact to reduce repeated input cost.\n\n"
    "  3. Model Routing  - Route simple definitional questions to a smaller /\n"
    "     cheaper/smaller Groq model when quality allows\n"
    "     and reserve openai/gpt-oss-20b for harder multi-step queries.\n\n"
    "  Combining these strategies can reduce projected monthly API spend by\n"
    "  roughly 40-60% while preserving answer quality for EduAssist.\n"
)


Average cost per request         : $0.00023797 USD
Average input tokens per request : 93.8
Average output tokens per request: 769.8

Projected Monthly Cost (10,000 requests):
  $2.3797 USD

Cost-Optimization Recommendation:

  1. Cap Output Length  - Adding max_tokens=300 for short factual questions
     bounds response length. On Groq, output tokens are priced higher than
     input tokens for this model, so shorter answers cut spend directly.

  2. Shorter System Prompts  - Keep stable role/instruction text minimal.
     v3's JSON schema adds input tokens on every call; keep the schema
     compact to reduce repeated input cost.

  3. Model Routing  - Route simple definitional questions to a smaller /
     cheaper/smaller Groq model when quality allows
     and reserve openai/gpt-oss-20b for harder multi-step queries.

  Combining these strategies can reduce projected monthly API spend by
  roughly 40-60% while preserving answer quality for EduAssist.



---
## Task 3 — Prompt Versioning and MLflow Evaluation  *(5 marks)*

**MLflow Experiment:** `eduassist_llmops_assignment`
**Prompt Registry Name:** `eduassist_prompt`
**Three prompt versions:** v1 (Basic) · v2 (Structured) · v3 (JSON Output)
**Metrics logged:** `avg_latency_seconds`, `avg_input_tokens`, `avg_output_tokens`, `avg_cost_per_request`, `valid_json`


In [9]:
# ── MLflow Experiment Setup ───────────────────────────────────────
# Start the MLflow UI in a separate terminal with:
#   mlflow ui --host 127.0.0.1 --port 5000

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f"✓ Tracking URI  : {MLFLOW_TRACKING_URI}")
print(f"✓ Experiment    : {EXPERIMENT_NAME}")
print(f"✓ Experiment ID : {experiment.experiment_id}")

mlflow_client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)


2026/11/03 20:58:45 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


✓ Tracking URI  : http://127.0.0.1:5000
✓ Experiment    : eduassist_llmops_assignment
✓ Experiment ID : 1


In [10]:
# ── Three Prompt Version Templates ──────────────────────────────
PROMPTS = {
    "v1": {
        "template": (
            "Answer the following student question clearly.\n\n"
            "Question: {{question}}"
        ),
        "commit_message": (
            "v1 – Basic: minimal instruction, no role or format constraint"
        ),
        "description": "Basic prompt — direct Q&A, no system role",
    },
    "v2": {
        "template": (
            "You are an academic assistant.\n"
            "Answer the following student question in a clear, concise, "
            "and accurate manner.\n"
            "Use simple language suitable for undergraduate students.\n\n"
            "Question: {{question}}"
        ),
        "commit_message": (
            "v2 – Structured: adds academic role, tone, and audience guidance"
        ),
        "description": "Structured prompt — role + style constraints",
    },
    "v3": {
        "template": (
            "You are an academic assistant.\n"
            "Answer the student question and return ONLY valid JSON.\n"
            "Do not include markdown, explanations outside JSON, or code fences.\n"
            "Use this exact JSON schema:\n"
            '{\n'
            '  "answer": "string",\n'
            '  "confidence": "high | medium | low",\n'
            '  "key_points": ["string"]\n'
            '}\n\n'
            "Question: {{question}}"
        ),
        "commit_message": (
            "v3 – JSON Output: enforces structured schema with confidence "
            "and key_points"
        ),
        "description": "JSON-output prompt — strict schema with confidence signal",
    },
}

for k, v in PROMPTS.items():
    print(f"\nPrompt {k}: {v['description']}")
    print(f"  Commit message: {v['commit_message']}")



Prompt v1: Basic prompt — direct Q&A, no system role
  Commit message: v1 – Basic: minimal instruction, no role or format constraint

Prompt v2: Structured prompt — role + style constraints
  Commit message: v2 – Structured: adds academic role, tone, and audience guidance

Prompt v3: JSON-output prompt — strict schema with confidence signal
  Commit message: v3 – JSON Output: enforces structured schema with confidence and key_points


In [11]:
# ── Register Prompts in MLflow Prompt Registry ───────────────────
# Requires MLflow >= 2.14.0 with Prompt Registry support.

# Clean slate for re-runs
try:
    mlflow_client.delete_registered_model(PROMPT_REGISTRY_NAME)
    print(f"Deleted existing '{PROMPT_REGISTRY_NAME}' for clean re-registration\n")
except Exception:
    pass  # Does not yet exist

print(f"Registering prompts under name '{PROMPT_REGISTRY_NAME}'...")
print("=" * 60)

registered_versions = {}
for key in ["v1", "v2", "v3"]:
    cfg = PROMPTS[key]
    try:
        # MLflow >= 2.14 Prompt Registry API
        prompt_obj = mlflow.register_prompt(
            name=PROMPT_REGISTRY_NAME,
            template=cfg["template"],
            commit_message=cfg["commit_message"],
            tags={
                "assignment_version": key,
                "description":        cfg["description"],
            },
        )
        registered_versions[key] = prompt_obj.version
        print(f"  ✓ {key} → MLflow prompt version {prompt_obj.version}")
        print(f"     URI: prompts:/{PROMPT_REGISTRY_NAME}/{prompt_obj.version}")
        print(f"     Commit: {cfg['commit_message']}")
    except AttributeError:
        # Fallback: log prompt template as a run artifact
        print(f"  ⚠  mlflow.register_prompt not found (MLflow < 2.14).")
        print(f"     Logging {key} template as a run artifact instead.")
        with mlflow.start_run(run_name=f"register_prompt_{key}"):
            mlflow.log_text(cfg["template"], f"prompt_{key}.txt")
            mlflow.log_param("prompt_version", key)
            mlflow.log_param("commit_message", cfg["commit_message"])
        registered_versions[key] = key

print(f"\n✓ Registered versions: {registered_versions}")


Registering prompts under name 'eduassist_prompt'...


2026/11/03 20:58:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_prompt, version 1


  ✓ v1 → MLflow prompt version 1
     URI: prompts:/eduassist_prompt/1
     Commit: v1 – Basic: minimal instruction, no role or format constraint


2026/11/03 20:58:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_prompt, version 2


  ✓ v2 → MLflow prompt version 2
     URI: prompts:/eduassist_prompt/2
     Commit: v2 – Structured: adds academic role, tone, and audience guidance


2026/11/03 20:58:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_prompt, version 3


  ✓ v3 → MLflow prompt version 3
     URI: prompts:/eduassist_prompt/3
     Commit: v3 – JSON Output: enforces structured schema with confidence and key_points

✓ Registered versions: {'v1': 1, 'v2': 2, 'v3': 3}


In [12]:
# ── Evaluate All 3 Prompt Versions on 5 Questions (15 calls) ─────
print("Running evaluations (15 API calls total)...")
print("=" * 60)

all_results = []

for version_key, cfg in PROMPTS.items():
    print(f"\nPrompt {version_key} — {cfg['description']}")
    for q in QUESTIONS:
        prompt_text = render_prompt(cfg["template"], q["question"])
        result      = call_llm(prompt_text, version_key)

        rec = {
            "question_id":     q["id"],
            "question":        q["question"],
            "prompt_version":  version_key,
            "model_name":      MODEL_NAME,
            "latency_seconds": result["latency_seconds"],
            "response_status": result["status"],
        }

        if result["response"]:
            u   = result["response"].usage
            txt = result["response"].choices[0].message.content
            ic, oc, tc = calculate_cost(u.prompt_tokens, u.completion_tokens)
            rec.update({
                "input_tokens":  u.prompt_tokens,
                "output_tokens": u.completion_tokens,
                "total_tokens":  u.total_tokens,
                "cost_usd":      tc,
                "response_text": txt,
            })
            if version_key == "v3":
                is_valid, _, msg = validate_json_response(txt)
                rec["is_valid_json"]  = int(is_valid)
                rec["json_parse_msg"] = msg
            else:
                rec["is_valid_json"]  = None
                rec["json_parse_msg"] = "N/A"
        else:
            rec.update({
                "input_tokens": 0, "output_tokens": 0, "total_tokens": 0,
                "cost_usd": 0.0,   "response_text": "",
                "is_valid_json":  0 if version_key == "v3" else None,
                "json_parse_msg": "API error",
            })

        all_results.append(rec)

        json_tag = (
            f"  JSON={'✓' if rec['is_valid_json'] else '✗'}"
            if version_key == "v3" else ""
        )
        print(f"  {q['id']}: {result['latency_seconds']:.3f}s | "
              f"{rec['total_tokens']} tok | ${rec['cost_usd']:.8f}{json_tag}")

df_results = pd.DataFrame(all_results)
print(f"\n✓ Evaluation complete — {len(df_results)} records")


Running evaluations (15 API calls total)...

Prompt v1 — Basic prompt — direct Q&A, no system role
  Q1: 0.708s | 659 tok | $0.00017655
  Q2: 1.035s | 690 tok | $0.00018585
  Q3: 1.155s | 864 tok | $0.00023737
  Q4: 1.280s | 722 tok | $0.00019545
  Q5: 1.651s | 1224 tok | $0.00034695

Prompt v2 — Structured prompt — role + style constraints
  Q1: 0.883s | 484 tok | $0.00011910
  Q2: 1.105s | 803 tok | $0.00021480
  Q3: 0.976s | 629 tok | $0.00016192
  Q4: 7.388s | 586 tok | $0.00014970
  Q5: 6.159s | 679 tok | $0.00017850

Prompt v3 — JSON-output prompt — strict schema with confidence signal
  Q1: 4.795s | 424 tok | $0.00009345  JSON=✓
  Q2: 4.902s | 369 tok | $0.00007695  JSON=✓
  Q3: 0.671s | 415 tok | $0.00009007  JSON=✓
  Q4: 0.699s | 375 tok | $0.00007875  JSON=✓
  Q5: 5.870s | 463 tok | $0.00010605  JSON=✓

✓ Evaluation complete — 15 records


In [13]:
# ── Log Per-Version Metrics to MLflow ───────────────────────────
print("Logging metrics to MLflow...")
print("=" * 60)

run_ids = {}
for version_key in ["v1", "v2", "v3"]:
    vd = df_results[df_results["prompt_version"] == version_key]

    avg_lat  = round(float(vd["latency_seconds"].mean()), 4)
    avg_in   = round(float(vd["input_tokens"].mean()),    2)
    avg_out  = round(float(vd["output_tokens"].mean()),   2)
    avg_cost = round(float(vd["cost_usd"].mean()),        8)

    valid_json_rate = None
    if version_key == "v3":
        valid_json_rate = round(
            float(vd["is_valid_json"].sum()) / len(vd), 4
        )

    with mlflow.start_run(run_name=f"prompt_{version_key}_eval") as run:
        run_ids[version_key] = run.info.run_id

        # ── Parameters ──────────────────────────────────────────
        mlflow.log_param("prompt_version",              version_key)
        mlflow.log_param("model_name",                  MODEL_NAME)
        mlflow.log_param("num_questions",               len(QUESTIONS))
        mlflow.log_param("pricing_date",                PRICING_DATE)
        mlflow.log_param("mlflow_prompt_registry_ver",
                         registered_versions.get(version_key, "N/A"))

        # ── Metrics ─────────────────────────────────────────────
        mlflow.log_metric("avg_latency_seconds",    avg_lat)
        mlflow.log_metric("avg_input_tokens",       avg_in)
        mlflow.log_metric("avg_output_tokens",      avg_out)
        mlflow.log_metric("avg_cost_per_request",   avg_cost)
        if valid_json_rate is not None:
            mlflow.log_metric("valid_json", valid_json_rate)

        # ── Prompt template as artifact ──────────────────────────
        template_path = f"/tmp/prompt_{version_key}.txt"
        with open(template_path, "w") as fh:
            fh.write(PROMPTS[version_key]["template"])
        mlflow.log_artifact(template_path, "prompts")

        line = (
            f"  {version_key} [run={run.info.run_id[:8]}...] "
            f"latency={avg_lat}s  in_tok={avg_in}  out_tok={avg_out}  "
            f"cost=${avg_cost:.8f}"
        )
        if valid_json_rate is not None:
            line += f"  valid_json={valid_json_rate:.0%}"
        print(line)

print("\n✓ All metrics logged")


Logging metrics to MLflow...
  v1 [run=93a8052a...] latency=1.1658s  in_tok=93.8  out_tok=738.0  cost=$0.00022843
🏃 View run prompt_v1_eval at: http://127.0.0.1:5000/#/experiments/1/runs/93a8052a5a0246f38f86290aadf59755
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
  v2 [run=543d4f42...] latency=3.3021s  in_tok=115.8  out_tok=520.4  cost=$0.00016480
🏃 View run prompt_v2_eval at: http://127.0.0.1:5000/#/experiments/1/runs/543d4f42170d4787b762c3550bd79d7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
  v3 [run=d1ca2583...] latency=3.3875s  in_tok=149.8  out_tok=259.4  cost=$0.00008905  valid_json=100%
🏃 View run prompt_v3_eval at: http://127.0.0.1:5000/#/experiments/1/runs/d1ca25831e444b948015a68682ae61ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

✓ All metrics logged


### Prompt Version Comparison Table

In [14]:
# ── Comparison Table and Best-Version Selection ──────────────────
comparison = []
for vk in ["v1", "v2", "v3"]:
    vd = df_results[df_results["prompt_version"] == vk]
    vj_str = (
        f"{int(vd['is_valid_json'].sum())}/{len(vd)} "
        f"({vd['is_valid_json'].sum()/len(vd):.0%})"
        if vk == "v3" else "N/A"
    )
    comparison.append({
        "Prompt Version":    vk,
        "Avg Latency (s)":   round(float(vd["latency_seconds"].mean()), 4),
        "Avg Input Tokens":  round(float(vd["input_tokens"].mean()),    1),
        "Avg Output Tokens": round(float(vd["output_tokens"].mean()),   1),
        "Avg Cost (USD)":    f"${float(vd['cost_usd'].mean()):.8f}",
        "Valid JSON Rate":   vj_str,
        "Recommended?":      "No",
    })

# Determine best version
v3_data            = df_results[df_results["prompt_version"] == "v3"]
v3_valid_json_rate = float(v3_data["is_valid_json"].sum()) / len(v3_data)

if v3_valid_json_rate >= 0.60:
    best_key, best_ver = "v3", 3
else:
    best_key, best_ver = "v2", 2

for row in comparison:
    if row["Prompt Version"] == best_key:
        row["Recommended?"] = "✓ PROMOTE"

df_compare = pd.DataFrame(comparison)
print(df_compare.to_string(index=False))
print(f"\n→ Best version: {best_key}  (will be promoted to 'production' alias)")
print(f"  v3 valid_json rate: {v3_valid_json_rate:.0%}")


Prompt Version  Avg Latency (s)  Avg Input Tokens  Avg Output Tokens Avg Cost (USD) Valid JSON Rate Recommended?
            v1           1.1658              93.8              738.0    $0.00022843             N/A           No
            v2           3.3021             115.8              520.4    $0.00016480             N/A           No
            v3           3.3875             149.8              259.4    $0.00008905      5/5 (100%)    ✓ PROMOTE

→ Best version: v3  (will be promoted to 'production' alias)
  v3 valid_json rate: 100%


In [15]:
# ── Promote Best Prompt to 'production' Alias ────────────────────
print(f"Promoting {best_key} (version {best_ver}) to 'production' alias...")

try:
    mlflow_client.set_registered_prompt_alias(
        name=PROMPT_REGISTRY_NAME,
        alias="production",
        version=best_ver,
    )
    print(f"✓ Alias set: prompts:/{PROMPT_REGISTRY_NAME}@production → version {best_ver}")
    print(f"  Load in future with:")
    print(f"  >>> mlflow.load_prompt('prompts:/{PROMPT_REGISTRY_NAME}@production')")
except Exception as exc:
    # Graceful fallback: tag the winning run
    print(f"  Note: set_registered_prompt_alias: {exc}")
    print(f"  Tagging the {best_key} MLflow run as production instead.")
    mlflow_client.set_tag(run_ids[best_key], "alias",                  "production")
    mlflow_client.set_tag(run_ids[best_key], "promoted_to_production",  "true")
    print(f"  ✓ Run {run_ids[best_key][:8]}... tagged as production")
    print(f"  In the MLflow UI: Registered Models → {PROMPT_REGISTRY_NAME}")
    print(f"  → version {best_ver} → Set Alias → type 'production'")


Promoting v3 (version 3) to 'production' alias...
  Note: set_registered_prompt_alias: 'MlflowClient' object has no attribute 'set_registered_prompt_alias'
  Tagging the v3 MLflow run as production instead.
  ✓ Run d1ca2583... tagged as production
  In the MLflow UI: Registered Models → eduassist_prompt
  → version 3 → Set Alias → type 'production'


In [16]:
# ── Promotion Justification (≥ 150 words) ────────────────────────
vj_pct = f"{v3_valid_json_rate:.0%}"
print(
    f"Promotion Justification — Prompt {best_key} -> 'production'\n"
    + "=" * 60 + "\n"
    f"Prompt {best_key} is promoted to the production alias in the EduAssist AI\n"
    "system for the following reasons:\n\n"
    "Structured, Machine-Parseable Output\n"
    "The JSON schema enforced by v3 — with 'answer', 'confidence', and\n"
    "'key_points' — makes downstream integration reliable and trivial.\n"
    "Unlike v1 and v2, v3 eliminates the need for ad-hoc text parsing,\n"
    "reduces integration brittleness, and enables automated features such\n"
    "as flashcard generation and topic tagging directly from the API response.\n\n"
    f"JSON Validity\n"
    f"Prompt v3 achieved a valid_json rate of {vj_pct} across five evaluation\n"
    "questions, demonstrating that the model reliably follows the schema.\n"
    "This rate is the primary quality gate for EduAssist's backend.\n\n"
    "Confidence Signalling for Safety\n"
    "The 'confidence' field (high | medium | low) provides immediate quality\n"
    "metadata per response. The system can escalate 'low'-confidence answers\n"
    "for human review without a separate classification step, improving\n"
    "educational safety without additional API costs.\n\n"
    "Latency and Cost Trade-off\n"
    "V3 has a slightly longer prompt than v1 or v2, increasing input tokens\n"
    "marginally. However, structured JSON with zero post-processing more than\n"
    "compensates. The cost difference per request is negligible at 10,000\n"
    "monthly requests.\n\n"
    "LLMOps Best Practices\n"
    "Versioning and objectively evaluating three prompt variants before\n"
    "promotion reflects sound LLMOps practice: prompts are treated as\n"
    "first-class production artefacts, measured on latency, token cost, and\n"
    "output quality. The 'production' alias decouples deployment from version\n"
    "numbers, enabling seamless rollback to v2 if v3 regresses without\n"
    "changing any downstream application code."
)


Promotion Justification — Prompt v3 -> 'production'
Prompt v3 is promoted to the production alias in the EduAssist AI
system for the following reasons:

Structured, Machine-Parseable Output
The JSON schema enforced by v3 — with 'answer', 'confidence', and
'key_points' — makes downstream integration reliable and trivial.
Unlike v1 and v2, v3 eliminates the need for ad-hoc text parsing,
reduces integration brittleness, and enables automated features such
as flashcard generation and topic tagging directly from the API response.

JSON Validity
Prompt v3 achieved a valid_json rate of 100% across five evaluation
questions, demonstrating that the model reliably follows the schema.
This rate is the primary quality gate for EduAssist's backend.

Confidence Signalling for Safety
The 'confidence' field (high | medium | low) provides immediate quality
metadata per response. The system can escalate 'low'-confidence answers
for human review without a separate classification step, improving
education

---
## Save CSV Outputs

In [17]:
# ── latency_tokens_cost.csv (Tasks 1 + 2) ────────────────────────
df_ltc = pd.DataFrame(task1_records)[[
    "question_id", "prompt_version", "model_name",
    "start_time", "end_time", "latency_seconds", "response_status",
    "input_tokens", "output_tokens", "total_tokens",
    "input_cost_usd", "output_cost_usd", "total_cost_usd",
]]
df_ltc.to_csv("latency_tokens_cost.csv", index=False)
print("✓ Saved latency_tokens_cost.csv")
print(df_ltc.to_string(index=False))

# ── prompt_evaluation_results.csv (Task 3) ────────────────────────
df_eval = df_results[[
    "question_id", "question", "prompt_version", "model_name",
    "latency_seconds", "input_tokens", "output_tokens", "total_tokens",
    "cost_usd", "is_valid_json", "json_parse_msg", "response_status",
]]
df_eval.to_csv("prompt_evaluation_results.csv", index=False)
print("\n✓ Saved prompt_evaluation_results.csv")
print(df_eval.to_string(index=False))


✓ Saved latency_tokens_cost.csv
question_id prompt_version         model_name                 start_time                   end_time  latency_seconds response_status  input_tokens  output_tokens  total_tokens  input_cost_usd  output_cost_usd  total_cost_usd
         Q1             v1 openai/gpt-oss-20b 2026-11-03T20:58:37.708023 2026-11-03T20:58:39.125783           1.4177         success            94            585           679        0.000007         0.000176        0.000183
         Q2             v1 openai/gpt-oss-20b 2026-11-03T20:58:39.125943 2026-11-03T20:58:41.270137           2.1442         success            94            877           971        0.000007         0.000263        0.000270
         Q3             v1 openai/gpt-oss-20b 2026-11-03T20:58:41.270325 2026-11-03T20:58:42.550925           1.2806         success            97            824           921        0.000007         0.000247        0.000254
         Q4             v1 openai/gpt-oss-20b 2026-11-03T20:58:42.55

---
## Assignment Summary

| Task | Deliverable | Status |
|---|---|---|
| Task 1 | Latency table + stats + interpretation | ✓ |
| Task 2 | Token/cost table + monthly projection + recommendation | ✓ |
| Task 3 | MLflow experiment + 3 prompt versions + metrics + comparison table + promotion | ✓ |
| CSVs | `latency_tokens_cost.csv`, `prompt_evaluation_results.csv` | ✓ |

**Provider:** Groq (`GROQ_API_KEY`) with model `openai/gpt-oss-20b`.

**MLflow UI:** Open `http://127.0.0.1:5000` to view the experiment, runs, Prompt Registry, and the `production` alias.

**Screenshots to capture for submission:**
1. MLflow Experiments page — showing `eduassist_llmops_assignment` runs
2. Run comparison view — metrics side-by-side for v1 / v2 / v3
3. Prompt Registry — showing v1, v2, v3 with commit messages
4. Prompt diff view — comparing any two versions
5. Production alias — `eduassist_prompt@production` pointing to the promoted version
